In [2]:
import os
os.chdir('../../../BertTopic/')

In [1]:
import re
import random
import pandas as pd
from tqdm.auto import tqdm
from hanziconv import HanziConv
from collections import defaultdict #当字典里的key不存在但被查找时，返回的不是keyError而是一个默认值

import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
train_filepath = './baseline/data/train.csv'
test_filepath = './baseline/data/test.csv'

train_ = pd.read_csv(train_filepath)
test_ = pd.read_csv(test_filepath)
total_ = pd.concat([train_, test_], axis=0)
# total_.to_csv('./total_data.csv', index=False, encoding='utf-8-sig')

In [48]:
len(total_)

7058

In [138]:
random.seed(80)

def data_split(data, rate=0.8):
    """
    训练集和测试集切分函数
    
    :param features: 输入的特征张量
    :param labels：输入的标签张量
    :param rate：训练集占所有数据的比例
    :return Xtrain, Xtest, ytrain, ytest：返回特征张量的训练集、测试集，以及标签张量的训练集、测试集 
    """
    num_examples = len(data)
    indices = list(range(num_examples))
    random.shuffle(indices)
    l = []
    num_train = int(num_examples * rate)
    indices_train = indices[: num_train]
    indices_test = indices[num_train: ]
    train_file = data.iloc[indices_train,:]
    test_file = data.iloc[indices_test,:]
    return train_file, test_file

train_file, test_file = data_split(total_, rate=0.8)
len(train_file), len(test_file)

(5646, 1412)

In [139]:
train_file.to_csv('./train.csv', index=False, encoding='utf-8-sig')
test_file.to_csv('./test.csv', index=False, encoding='utf-8-sig')

In [2]:
def data_access(filepath): 
    """数据获取与预处理"""
    raw_iter = pd.read_csv(filepath)
#     raw_iter['label'].value_counts()
    data = []
    for raw in raw_iter.values:
        label, s = raw[-1], raw[1]  # 标签和文本 
        s = re.sub(r'http://.*?(\s|$)', '<URL>'+'\\1', s)
        s = re.sub(r'@.*?(\s|$)', '<@ID>'+'\\1', s)
        s = re.sub('\u200B', '', s)
        s = HanziConv.toSimplified(s.strip())
        data.append((s,label))
    return data

train_filepath = '../BertTopic/data/train.csv'
test_filepath = '../BertTopic/data/test.csv'
train_data = data_access(train_filepath)
test_data = data_access(test_filepath)

# filepath = './total_data.csv'
# total_data = data_access(filepath)
# len(total_data)

In [3]:
# total_data[0]
len(train_data), len(test_data)

(5646, 1412)

In [4]:
def sample_count(data):
    """观察划分后的训练集和测试集中正负样本的比例"""
    pos = 0
    for index in data:
        s,l = index[0], index[1]
        if int(l) == 1:
            pos +=1
    neg = len(data) - pos
    return pos, neg

train_pos, train_neg =sample_count(train_data)
test_pos, test_neg = sample_count(test_data)
print(f'训练集中正样本为{train_pos}条，负样本为{train_neg}条。')
print(f'测试集中正样本为{test_pos}条，负样本为{test_neg}条。')

训练集中正样本为2580条，负样本为3066条。
测试集集中正样本为616条，负样本为796条。


In [5]:
from collections import defaultdict #当字典里的key不存在但被查找时，返回的不是keyError而是一个默认值

class Vocab:
    def __init__(self, tokens=None):
        self.idx_to_token = list() #词表
        self.token_to_idx = dict() #词表及对应单词位置

        if tokens is not None:
            if "<unk>" not in tokens:
                tokens = tokens + ["<unk>"]
            for token in tokens:
                self.idx_to_token.append(token)
                self.token_to_idx[token] = len(self.idx_to_token) - 1 #标记每个单词的位置
            self.unk = self.token_to_idx['<unk>'] #开始符号的位置
    
    @classmethod 
    #不需要实例化，直接类名.方法名()来调用 不需要self参数，但第一个参数需要是表示自身类的cls参数,
    #因为持有cls参数，可以来调用类的属性，类的方法，实例化对象等    
    def build(cls, text, min_freq=1, reserved_tokens=None):
        token_freqs = defaultdict(int)
        for sentence in text:
            for token in sentence:
                token_freqs[token] += 1
        uniq_tokens = ["<unk>"] + (reserved_tokens if reserved_tokens else [])
        uniq_tokens += [token for token, freq in token_freqs.items() \
                        if freq >= min_freq and token != "<unk>"]
        return cls(uniq_tokens)
    
    def __len__(self):
        #返回词表的大小，即词表中有多少个互不相同的标记
        return len(self.idx_to_token)
    def __getitem__(self,token):
        #查找输入标记对应的索引值，如果该标记不存在，则返回标记<unk>的索引值（0）
        return self.token_to_idx.get(token,self.unk)
    def convert_tokens_to_ids(self,tokens):
        #查找一系列输入标记对应的索引值
        return [self[token] for token in tokens]
    def convert_ids_to_tokens(self,indices):
        #查找一系列索引值对应的标记
        return [self.idx_to_token[index] for index in indices]

In [6]:
class FakeNewsDataset(Dataset):
    def __init__(self,dataset):
        self.dataset = dataset
        self.lens = len(dataset)
    def __getitem__(self,index):
        sen,label = self.dataset[index]
        return sen,label
    def __len__(self):
        return len(self.dataset)
    
def collate_fn_textcnn(examples):
    """对一个批次内的数据进行处理"""
    inputs = [torch.tensor(ex[0]) for ex in examples]
    tragets = torch.tensor([ex[1] for ex in examples], dtype=torch.long)
    # 对批次内的样本进行补齐，使其具有相同长度
    inputs = pad_sequence(inputs, batch_first=True)
    return inputs, tragets

In [7]:
class TextCNN(nn.Module):
    '''
    vocab_size: 词表大小
    embedding_dim: 经过embedding转换后词向量的维度
    filter_size: 卷积核的大小
    num_filter: 卷积核的个数
    num_class：类别数
    '''
    def __init__(self, vocab_size, embedding_dim, filter_size, num_filter, num_class):
        super(TextCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # padding=1 表示在卷积操作之前，将序列的前后各补充1个输入，这里没有找到详细的解释，考虑是为了让卷积核充分学习序列的信息
        self.conv1d = nn.Conv1d(embedding_dim, num_filter, filter_size, padding=1)  
        self.activate = F.relu
        self.linear = nn.Linear(num_filter, num_class)

    def forward(self, inputs):
        # 输入数据维度 （batch_size, max_src_len）
        embedding = self.embedding(inputs)
        # 经过embedding层后，维度变为（batch_size, max_src_len, embedding_dim）
        # 但卷积层要求输入数据的形状为（batch_size, in_channels, max_src_len）,因此这里对输入数据进行维度交换
        # embedding.permute(0,2,1) 交换embedding中维度2和1的位置
        convolution = self.activate(self.conv1d(embedding.permute(0, 2, 1))) 
        # 经过卷积层输出以后的维度为（batch_size, out_channels, out_seq_len）
        pooling = F.max_pool1d(convolution, kernel_size=convolution.shape[2])  # 池化层聚合
        # 经过池化层输出以后的维度为（batch_size,out_channels,1）
        # 但由于全连接层要求输入数据的最后一个维度为卷积核的个数，即out_channels,因此这里需要降维
        probs = self.linear(pooling.squeeze(dim=2))
        return probs

In [8]:
def evaluate(model, data_loader, device):
    test_acc = 0
    test_loss = 0
    model.eval()  # 切换到测试模式
    with torch.no_grad():  # 不计算梯度
        y_true, y_pred = [], []
        for batch in data_loader:
            inputs, targets = [x.to(device) for x in batch]
            probs = model(inputs)
            loss = creterion(probs, targets)
            acc = (probs.argmax(dim=1) == targets).sum().item()
            test_acc += acc
            test_loss += loss.item()
            
            batch_pred = probs.argmax(dim=1).tolist()  # 得到一个batch的预测标签
            batch_true = targets.tolist()
             
            y_pred.extend(batch_pred)
            y_true.extend(batch_true)
        print(f'Test Loss:{test_loss:.4f}\tTest Accuracy:{test_acc/len(test_dataset):.4f}')
        acc = accuracy_score(y_true, y_pred)
        p = precision_score(y_true, y_pred)
        r = recall_score(y_true, y_pred)
        f = f1_score(y_true, y_pred)
    model.train()  # 切换到训练模式
    return acc, p, r, f, y_true, y_pred   

In [43]:
torch.manual_seed(111)

# 数据集划分 确定训练集、测试集大小，以8：2划分训练集和测试集
num_train = int(len(total_data) * 0.8)
num_test = len(total_data) - num_train
train_data, test_data = random_split(total_data, [num_train, num_test])

def sample_count(data_total,data_splited):
    """观察划分后的训练集和测试集中正负样本的比例"""
    pos = 0
    for index in data_splited.indices:
        s,l = data_total[index]
        if int(l) == 1:
            pos +=1
    neg = len(data_splited) - pos
    return pos, neg

train_pos, train_neg =sample_count(total_data, train_data)
test_pos, test_neg = sample_count(total_data, test_data)
print(f'训练集中正样本为{train_pos}条，负样本为{train_neg}条。')
print(f'测试集中正样本为{test_pos}条，负样本为{test_neg}条。')

训练集中正样本为2573条，负样本为3073条。
测试集中正样本为653条，负样本为759条。


In [9]:
# 根据训练集进行词表映射
train_sentences = [s for s,l in train_data]
vocab = Vocab.build(train_sentences)  
len(vocab)

4247

In [10]:
# 词表映射
train_data = [(vocab.convert_tokens_to_ids(list(sentence)), label) for sentence,label in train_data]
test_data = [(vocab.convert_tokens_to_ids(list(sentence)), label) for sentence,label in test_data]

In [23]:
torch.manual_seed(100)
config = {'num_class':2,
          'batch_size':128,
          'learning_rate':0.03,
          'num_epochs':10,
          'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu')
         }

train_dataset = FakeNewsDataset(train_data)
test_dataset = FakeNewsDataset(test_data)
train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, collate_fn=collate_fn_textcnn)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, collate_fn=collate_fn_textcnn)

model = TextCNN(vocab_size=len(vocab), embedding_dim=256, filter_size=3, num_filter=64, num_class=2) 
model.to(config['device'])
creterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=config['learning_rate'])

In [24]:
for epoch in range(config['num_epochs']):
    total_loss = 0
    total_acc = 0
    for batch in tqdm(train_loader, desc=f'Training Epoch {epoch+1}'):
        # 将数据加载至GPU
        inputs, targets = [x.to(config['device']) for x in batch]
        # 将特征带入到模型
        probs = model(inputs)
        # 计算损失
        loss = creterion(probs, targets)
        optimizer.zero_grad()  # 梯度清零
        loss.backward()  # 反向传播
        optimizer.step()  # 更新参数
        acc = (probs.argmax(dim=1) == targets).sum().item()  # item()用于在只包含一个元素的tensor中提取值
        total_acc += acc  # 最终得到整个epoch的准确率
        total_loss += loss.item()  # 最终得到整个epoch的损失
        
    # 打印的是整个eopch上的样本损失的平均值以及准确率
    print(f'Train Loss:{total_loss:.4f}\tTrain Accuracy:{total_acc/len(train_dataset):.4f}')
    acc, p, r, f, y_true, y_pred = evaluate(model, test_loader, config['device'])
    print(f"Test Results:    Precision: {p:.4f},  Recall: {r:.4f},  F1: {f:.4f}")

Training Epoch 1:   0%|          | 0/45 [00:00<?, ?it/s]

Train Loss:22.1946	Train Accuracy:0.7481
Test Loss:7.5086	Test Accuracy:0.6112
Test Results:    Precision: 0.5303,  Recall: 0.9529,  F1: 0.6814


Training Epoch 2:   0%|          | 0/45 [00:00<?, ?it/s]

Train Loss:8.6991	Train Accuracy:0.9401
Test Loss:4.4179	Test Accuracy:0.8711
Test Results:    Precision: 0.8546,  Recall: 0.8490,  F1: 0.8518


Training Epoch 3:   0%|          | 0/45 [00:00<?, ?it/s]

...

Train Loss:0.9855	Train Accuracy:0.9977
Test Loss:4.6577	Test Accuracy:0.8895
Test Results:    Precision: 0.8966,  Recall: 0.8442,  F1: 0.8696


Training Epoch 9:   0%|          | 0/45 [00:00<?, ?it/s]

Train Loss:0.8333	Train Accuracy:0.9977
Test Loss:5.0422	Test Accuracy:0.8895
Test Results:    Precision: 0.9137,  Recall: 0.8247,  F1: 0.8669


Training Epoch 10:   0%|          | 0/45 [00:00<?, ?it/s]

Train Loss:0.7182	Train Accuracy:0.9982
Test Loss:5.1718	Test Accuracy:0.8867
Test Results:    Precision: 0.9116,  Recall: 0.8198,  F1: 0.8632
